# 03. Siniestralidad y conglomerados

In [113]:
# ------------------------------------------
# 1. Importación de librerías
# ------------------------------------------

from pathlib import Path
import sys
import pandas as pd
import plotly.express as px
from IPython.display import display

# ------------------------------------------
# 2. Configuración de rutas
# ------------------------------------------

# Localizar la carpeta principal del proyecto
ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "funciones_reservas.py").exists()
)

# Permitir importar nuestras funciones
sys.path.insert(0, str(ROOT / "src"))

from funciones_reservas import leer_snapshot

# Definir las carpetas de trabajo
PROCESSED = ROOT / "data" / "processed"

# ------------------------------------------
# 3. Cargar las bases ajustadas
# ------------------------------------------

# Recuperar las bases previamente ajustadas a diciembre de 2025
emision_ajustada = leer_snapshot(
    PROCESSED / "emision_ajustada_2025.parquet"
)

siniestros_ajustados = leer_snapshot(
    PROCESSED / "siniestros_ajustados_2025.parquet"
)

# Mostrar las dimensiones de las bases
print(f"Emisión: {emision_ajustada.shape}")
print(f"Siniestros: {siniestros_ajustados.shape}")


Emisión: (21353, 11)
Siniestros: (2763, 14)


# 3.1. Preparación y cálculo de factores de siniestralidad

## 3.1.1. Cálculo de factores de siniestralidad por año de origen

Qué tan grandes son los siniestros en relación con la prima emitida por año

In [114]:
# ------------------------------------------
# 3.1.1 Siniestralidad total por año 
# ------------------------------------------

# Columnas de desarrollo ajustadas por INPC
columnas = [f"{i}_monto pagado_ajustada" for i in range(4)]

# Sumar los desarrollos de cada registro
siniestros_ajustados["siniestro_total"] = (
    siniestros_ajustados[columnas].sum(axis=1, min_count=1)
)

# Agrupar los siniestros por año de origen
siniestros_anual = (
    siniestros_ajustados
    .groupby("año")["siniestro_total"]
    .sum()
).round(2)

# Sumar todas las primas emitidas de cada año
primas_anual = (
    emision_ajustada
    .groupby("año")["prima_emitida_ajustada"]
    .sum()
).round(2)

# Calcular el factor de siniestralidad del mercado
factor_anual = siniestros_anual / primas_anual.replace(0, 1)

factor_anual.reset_index().rename(columns={0: "factor_siniestralidad_anual"}).set_index("año")

,factor_siniestralidad_anual
año,
2015,1.728247
2016,0.553433
2017,0.336023
2018,0.376379
2019,0.382350
2020,0.638191
2021,0.519178
2022,0.672990
2023,0.442951


In [115]:
# ------------------------------------------
# Métricas de siniestralidad por año
# ------------------------------------------
    
print("Métricas de siniestralidad anual")
factor_anual.describe()


Métricas de siniestralidad anual


count    11.000000
mean      0.611773
std       0.395429
min       0.336023
25%       0.379365
50%       0.519178
75%       0.655591
max       1.728247
dtype: float64

In [116]:
# ------------------------------------------------------
# Evolución del factor de siniestralidad anual por año
# ------------------------------------------------------

fig = px.line(
    factor_anual, 
    x=factor_anual.index,
    y=factor_anual.values,
    markers=True,
    title="Evolución del factor de siniestralidad anual",
    labels={"x": "Año", "y": "Factor de siniestralidad"},
)

fig.update_layout(
    hovermode="x unified",
    legend_title_text = "Indicator"
)

fig.show()

El factor de siniestralidad del mercado presenta fluctuaciones importantes entre 2015 y 2025, destacando 2015 como el año con mayor siniestralidad relativa (1.72), mientras que en los años posteriores los factores se mantienen por debajo de 1, este comportamiento sugiere que la siniestralidad no ha seguido una tendencia constante y que podrían existir diferencias importantes entre los segmentos del mercado, por lo que resulta conveniente analizar sus factores individuales para identificar cuáles presentan mayor siniestralidad y variabilidad.

## 3.1.2. Calculo de factores de siniestralidad por tipo de embarcación y uso (perfil de riesgo) en cada año de origen

Qué tan grandes son lo siniestros de cada perfil de riesgo en razón con la prima emitida

In [ ]:
# ------------------------------------------
# 3.1.2 Siniestralidad total por año 
# ------------------------------------------

# Sumar los desarrollos de cada registro
siniestros_ajustados["siniestro_total"] = (
    siniestros_ajustados[columnas].sum(axis=1, min_count=1)
)

# Agrupar los siniestros por año de origen, tipo de embarcación y uso
siniestros_segmento = (
    siniestros_ajustados
    .groupby(["año", "tipo_embarcacion", "uso"])["siniestro_total"]
    .sum()
    .round(2)
)

# Sumar todas las primas emitidas de cada año, tipo de embarcación y uso
primas_anual_tipo = (
    emision_ajustada
    .groupby(["año", "tipo_embarcacion", "uso"])["prima_emitida_ajustada"]
    .sum()
).round(2)

# Calcular el factor de siniestralidad del mercado
factor_anual_tipo = siniestros_segmento / primas_anual_tipo.replace(0, 1)

print("Factor de siniestralidad anual por tipo de embarcación y uso")

factor_anual_tipo.reset_index().rename(columns={0: "factor_siniestralidad_segmento"}).set_index(["año", "tipo_embarcacion", "uso"])

Factor de siniestralidad anual por tipo de embarcación y uso


factor_siniestralidad_segmento
año  tipo_embarcacion uso                                               
2015 abastecedor      abastecedor                                    NaN
                      comercial                                 0.262124
                      otro uso                                  9.549007
                      pesca                                     0.166331
                      privado y/o placer                             NaN
...                                                                  ...
2025 velero           pesca                                          NaN
                      privado y/o placer                        0.912612
                      turistico                                 0.045509
     yate             privado y/o placer                        1.669954
                      turistico                                 0.516606

[885 rows x 1 columns]

In [118]:
# ------------------------------------------
# 3. Identificación de segmentos
# ------------------------------------------

# Convertir las primas agrupadas en un DataFrame
emision_segmentos = primas_anual_tipo.reset_index()

# Identificar combinaciones únicas de tipo y uso
segmentos = emision_segmentos[
    ["tipo_embarcacion", "uso"]
].drop_duplicates()

print("Número de segmentos:", len(segmentos))

# --------------------------------------------------
# Cobertura de segmentos durante 2015-2022
# --------------------------------------------------

# Seleccionar los años con desarrollo completo
periodo = emision_segmentos[
    emision_segmentos["año"].between(2015, 2022)
]

# Contar años con prima positiva por segmento
cobertura_8 = (
    periodo
    .assign(positiva=periodo["prima_emitida_ajustada"] > 0)
    .groupby(["tipo_embarcacion", "uso"])["positiva"]
    .sum()
)

# Identificar segmentos con prima positiva en los 8 años
segmentos_completos = cobertura_8[cobertura_8 == 8].index

print("Segmentos completos 2015-2022:", len(segmentos_completos))

# --------------------------------------------------
# Participación de los segmentos seleccionados
# --------------------------------------------------

# Identificar los 34 segmentos dentro de 2015-2022
seleccion = periodo.set_index(
    ["tipo_embarcacion", "uso"]
).index.isin(segmentos_completos)

# Calcular las primas emitidas del periodo
prima_mercado = periodo["prima_emitida_ajustada"].sum()
prima_segmentos = periodo.loc[
    seleccion, "prima_emitida_ajustada"
].sum()

# Calcular su participación dentro del mercado
participacion = prima_segmentos / prima_mercado * 100

print(f"Participación de los 34 segmentos: {participacion:.2f}%")

Número de segmentos: 132
Segmentos completos 2015-2022: 34
Participación de los 34 segmentos: 89.76%


Para el análisis de conglomerados se consideraron las combinaciones de tipo de embarcación y uso, identificando inicialmente 132 segmentos. Se seleccionaron 34 con primas emitidas positivas durante todos los años de 2015 a 2022, los cuales representan el 89.76% de las primas emitidas del periodo. Esta selección permite trabajar con trayectorias de siniestralidad temporalmente comparables y con una parte importante del mercado.

## 3.1.3. Análisis descriptivo: evolución, extremos y variabilidad

In [130]:
# ---------------------------------------------------------------------
# Dataframe del factor de siniestralidad de los segmentos seleccionados
# ---------------------------------------------------------------------

# Convertir los factores en DataFrame
df_factores = factor_anual_tipo.rename("factor_siniestralidad").reset_index()

# Identificar los 34 segmentos seleccionados
filtro = df_factores.set_index(
    ["tipo_embarcacion", "uso"]
).index.isin(segmentos_completos)

# Filtrar los segmentos y años de interés
df_34 = df_factores.loc[
    filtro & df_factores["año"].between(2015, 2022)
].copy()

df_34["factor_siniestralidad"] = (df_34["factor_siniestralidad"].fillna(0))

display(df_34)

,año,tipo_embarcacion,uso,factor_siniestralidad
0,2015,abastecedor,abastecedor,0.000000
1,2015,abastecedor,comercial,0.262124
4,2015,abastecedor,privado y/o placer,0.000000
7,2015,abastecedor,turistico,0.000000
10,2015,atunero,pesca,0.045858
...,...,...,...,...
646,2022,velero,pesca,0.000000
647,2022,velero,privado y/o placer,0.730690
648,2022,velero,turistico,2.638522
649,2022,yate,privado y/o placer,0.218107


Se identificaron perfiles de riesgo que, a pesar de presentar primas emitidas, no registraron siniestros durante determinados años, por lo que se les asignó un factor de siniestralidad igual a cero, ya que no existe un monto de siniestros que representar respecto a sus primas.

# 3.2. Preprocesamiento multivariado


## 3.2.1. Formulación del problema y construcción de la matriz

- ¿Qué segmentos presentan mayor siniestralidad relativa?
- ¿Cuáles tienen factores más estables?
- ¿Qué segmentos registran años con pérdidas extraordinarias?
- ¿Existen segmentos con comportamientos históricos similares?


## 3.2.2. Análisis de correlaciones y selección de variables



## 3.2.3. Tratamiento de valores extremos, faltantes y estandarización


## 3.3. Análisis exploratorio multivariado (EDA)

## 3.3.1. PCA: varianza explicada, scree plot y loadings

## 3.3.2. Círculo de correlaciones y biplot

## 3.3.3. Curvas de Andrews

## 3.3.4. Conclusiones del EDA


## 3.4. Análisis de conglomerados

## 3.4.1. Selección de la distancia

## 3.4.2. Aplicación de métodos de agrupación

## 3.4.3. Determinación del número de conglomerados (k)

## 3.4.4. Asignación y caracterización de los grupos


## 3.5. Evaluación de los conglomerados

## 3.5.1. Silhouette Score

## 3.5.2. Calinski-Harabasz y WCSS

## 3.5.3. Comparación de métodos y tamaños de grupos


## 3.6. Validación de estabilidad

## 3.6.1. Comparación de métodos mediante ARI

## 3.6.2. Sensibilidad a valores extremos y selección de años

## 3.6.3. Estabilidad temporal


## 3.7. Conclusiones

## 3.7.1. Perfiles de siniestralidad identificados

## 3.7.2. Calidad, estabilidad y limitaciones de los grupos